## Cài đặt và Kiểm chứng các phép đo Thống kê & Cross-Validation cho OLS
Phần này thực hiện chẩn đoán mô hình, suy diễn hệ số ($t$-test, $F$-test), phân tích phần dư và K-fold CV. 
Các công thức toán học chi tiết (như ma trận hiệp phương sai, phân rã $TSS = ESS + RSS$) được chứng minh trong file Báo cáo (PDF).

In [ ]:
import numpy as np
import statsmodels.api as sm
from ols_implementation import model_metrics, coef_inference, gauss_markov_simulation
from residual_analysis import residual_plots
from cross_validation import kfold_cv

# Tạo Toy Data (Có ý nghĩa toán học)
np.random.seed(42)
n, p = 100, 2
X_toy = np.random.randn(n, p)
X_toy_design = np.c_[np.ones(n), X_toy] # Thêm Intercept
true_beta = np.array([5.0, 2.0, -3.0])
y_toy = X_toy_design.dot(true_beta) + np.random.randn(n) * 1.5 # Noise ~ N(0, 1.5^2)

# Tính nhanh beta_hat bằng Numpy để làm đầu vào cho các hàm
beta_hat = np.linalg.inv(X_toy_design.T.dot(X_toy_design)).dot(X_toy_design.T).dot(y_toy)
y_hat = X_toy_design.dot(beta_hat)
residuals = y_toy - y_hat
sigma2 = np.sum(residuals**2) / (n - (p + 1))

### 1. Suy diễn hệ số ($t$-test) và Thống kê tổng quát ($F$-test, $R^2$)
Kiểm tra tính chính xác của hàm `model_metrics` và `coef_inference` so với thư viện `statsmodels`.

**1. Đánh giá tổng thể mô hình:**
* **Vector phần dư (Residuals):** $e = y - \hat{y}$
* **Tổng bình phương phần dư (RSS):** $RSS = \sum_{i=1}^n (y_i - \hat{y}_i)^2 = e^T e$
* **Vector trung tâm (Centered Target):** $y_c = y - \bar{y}$
* **Tổng bình phương toàn phần (TSS):** $TSS = \sum_{i=1}^n (y_i - \bar{y})^2 = y_c^T y_c$
* **Hệ số xác định ($R^2$):** $R^2 = 1 - \frac{RSS}{TSS}$
* **$R^2$ hiệu chỉnh:** $R^2_{adj} = 1 - \left( \frac{RSS / (n - p - 1)}{TSS / (n - 1)} \right)$
* **Kiểm định $F$:** $F = \frac{(TSS - RSS) / p}{RSS / (n - p - 1)}$

**2. Suy diễn hệ số (Inference):**
* **Ma trận hiệp phương sai:** $\text{Var}(\hat{\beta}) = \hat{\sigma}^2 (X^T X)^{-1}$ với $\hat{\sigma}^2 = \frac{RSS}{n - p - 1}$
* **Sai số chuẩn ($SE$):** $SE(\hat{\beta}_j) = \sqrt{\text{Var}(\hat{\beta})_{jj}}$
* **Kiểm định $t$:** $t_j = \frac{\hat{\beta}_j}{SE(\hat{\beta}_j)}$
* **Khoảng tin cậy 95%:** $CI = \hat{\beta}_j \pm t_{\alpha/2, n-p-1} \times SE(\hat{\beta}_j)$

In [ ]:
metrics = model_metrics(y_toy, y_hat, p)
inference_df = coef_inference(X_toy_design, y_toy, beta_hat, sigma2)

print(f"RSS          : {metrics['RSS']:.4f}")
print(f"TSS          : {metrics['TSS']:.4f}")
print(f"R-squared    : {metrics['R2']:.4f}")
print(f"Adj R-squared: {metrics['Adj_R2']:.4f}")
print(f"F-statistic  : {metrics['F_stat']:.4f}")
print(f"F p-value    : {metrics['F_pvalue']:.4e}\n")
display(inference_df)

In [ ]:
print("Kiểm chứng với statmodels")
sm_model = sm.OLS(y_toy, X_toy_design).fit()
print(sm_model.summary().tables[0])
print(sm_model.summary().tables[1])

* **Nhận xét:** Các chỉ số $R^2$, $F$-statistic, Standard Errors, và $p$-value từ hàm tự code khớp chính xác 100% với output của `statsmodels` (đến chữ số thập phân thứ 4). Điều này chứng minh việc cài đặt các công thức đại số tuyến tính và tra bảng phân phối Student/Fisher bằng `scipy` là hoàn toàn chính xác.

### 2. Phân tích phần dư & Đánh giá giả định Gauss-Markov

Để kiểm chứng toàn diện, ta vẽ 4 biểu đồ chẩn đoán chuẩn mực (tương đương `plot(lm)` trong R) trên 2 kịch bản dữ liệu giả lập:

**Công thức Toán học cài đặt:**
* **Giá trị đòn bẩy (Leverage):** $h_{ii}$ là các phần tử trên đường chéo của Hat Matrix $H = X(X^T X)^{-1}X^T$.
* **Phần dư chuẩn hóa (Standardized Residuals):** $r_i = \frac{e_i}{\hat{\sigma} \sqrt{1 - h_{ii}}}$
* **Q-Q Plot:** Sử dụng phép xấp xỉ Tukey Lambda để tính phân vị lý thuyết, không phụ thuộc vào thư viện `scipy`.

**Kịch bản 1: Dữ liệu chuẩn (Phương sai không đổi)**
* *Nhận xét Biểu đồ:* * **Residuals vs Fitted:** Các điểm phân tán ngẫu nhiên quanh trục 0 $\rightarrow$ Thỏa mãn tính tuyến tính.
    * **Normal Q-Q:** Bám sát đường chéo đỏ $\rightarrow$ Phần dư tuân theo phân phối chuẩn.
    * **Scale-Location:** Đường xu hướng nằm ngang $\rightarrow$ Thỏa mãn Homoscedasticity (Phương sai đồng đều).
    * **Residuals vs Leverage:** Không có điểm nào nằm tách biệt ở góc phải (Leverage cao) với phần dư lớn $\rightarrow$ Không có điểm ảnh hưởng xấu (Influential Outliers).

**Kịch bản 2: Dữ liệu lỗi (Phương sai thay đổi - Heteroscedasticity)**
* *Nhận xét Biểu đồ:* * **Scale-Location:** Biểu đồ có hình cái phễu (phân tán rộng dần về bên phải) $\rightarrow$ Vi phạm giả định phương sai không đổi. Mô hình OLS tuy không chệch nhưng mất đi tính hiệu quả tối ưu (không còn là BLUE).

In [ ]:
# Case 1: Dùng Toy Data
print("KỊCH BẢN 1: Phương sai sai số không đổi")
residual_plots(X_toy_design, y_toy, beta_hat)

In [ ]:
# Case 2: Tạo Bad Data
noise_hetero = np.random.randn(n) * (X_toy[:, 0] ** 2) * 3  # Nhiễu phình to theo X1
y_bad = X_toy_design.dot(true_beta) + noise_hetero
beta_bad = (
    np.linalg.inv(X_toy_design.T.dot(X_toy_design)).dot(X_toy_design.T).dot(y_bad)
)

print("KỊCH BẢN 2: Phương sai sai số thay đổi (Heteroscedasticity)")
residual_plots(X_toy_design, y_bad, beta_bad)

* **Nhận xét Kịch bản 1:** Biểu đồ Normal Q-Q bám sát đường chéo, Scale-Location nằm ngang. Thỏa mãn các giả định Gauss-Markov.
* **Nhận xét Kịch bản 2:** Biểu đồ Scale-Location có hình cái phễu (toe ra ở đuôi). Suy ra hiện tượng Heteroscedasticity. Mô hình OLS lúc này dù không chệch nhưng không còn là BLUE (Best Linear Unbiased Estimator).

### 3. K-Fold Cross-Validation: Đánh giá mô hình & Phát hiện Overfitting

**1. Phân chia dữ liệu (Data Splitting):**
* Tập dữ liệu gốc có kích thước $n$ được chia thành $k$ phần (folds) rời rạc có kích thước xấp xỉ nhau: $S_1, S_2, \dots, S_k$.

**2. Vòng lặp K-Fold (Với mỗi fold $i$ từ $1$ đến $k$):**
* **Phân tách tập:** Chọn phần $S_i$ làm tập kiểm định (Validation set) chứa $m$ mẫu. Các phần còn lại $S \setminus S_i$ tạo thành tập huấn luyện (Train set).
* **Huấn luyện mô hình:** Ước lượng vector hệ số $\hat{\beta}^{(i)}$ trên tập Train bằng phương pháp OLS.
* **Dự đoán:** Tính vector giá trị dự đoán trên tập Validation: 
  $$\hat{y}_{val}^{(i)} = X_{val}^{(i)} \hat{\beta}^{(i)}$$
* **Tính lỗi của fold (Mean Squared Error - MSE):**
  $$MSE_i = \frac{1}{m} \sum_{j=1}^{m} \left( y_{val, j}^{(i)} - \hat{y}_{val, j}^{(i)} \right)^2$$

**3. Điểm đánh giá tổng hợp (CV Score):**
* Lỗi K-Fold CV là trung bình cộng của lỗi trên tất cả $k$ folds:
  $$CV\_Score = \frac{1}{k} \sum_{i=1}^k MSE_i$$

In [ ]:
# Tạo biến rác
X_garbage = np.random.randn(n, 5)
X_overfit = np.c_[X_toy, X_garbage]

# Chạy K-Fold (k=5)
cv_score_good = kfold_cv(X_toy, y_toy, k=5)
cv_score_overfit = kfold_cv(X_overfit, y_toy, k=5)

print(f"Mô hình 1 (Chỉ dùng biến đúng): CV RMSE = {cv_score_good:.4f}")
print(f"Mô hình 2 (Nhét thêm 5 biến rác): CV RMSE = {cv_score_overfit:.4f}")

* **Nhận xét:** Mô hình 2 bị Overfitting nên điểm CV RMSE kém hơn hẳn (cao hơn) Mô hình 1, dù trên tập Train nó có thể có RSS nhỏ hơn. Hàm `kfold_cv` tự cài đặt đã hoạt động đúng bản chất trong việc đánh giá năng lực tổng quát hóa của mô hình.

### 4. Mô phỏng Monte Carlo định lý Gauss-Markov

In [ ]:
gauss_markov_simulation()